In [11]:
pip install databento pandas numpy matplotlib


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/pty.py:66: DeprecationWarning: This process (pid=10480) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Note: you may need to restart the kernel to use updated packages.


In [8]:
import databento as db
import pandas as pd

# 1. Load just ONE day of data
path = "data/raw/xnas-itch-20230801.mbp-1.dbn.zst"
store = db.DBNStore.from_file(path)

# 2. Convert to a Pandas DataFrame
# Note: MBP-1 data includes both trades and quote updates
df = store.to_df()

# 3. Look at the columns
print(df.columns)
print(df.head())

Index(['ts_event', 'rtype', 'publisher_id', 'instrument_id', 'action', 'side',
       'depth', 'price', 'size', 'flags', 'ts_in_delta', 'sequence',
       'bid_px_00', 'ask_px_00', 'bid_sz_00', 'ask_sz_00', 'bid_ct_00',
       'ask_ct_00', 'symbol'],
      dtype='str')
                                                               ts_event  \
ts_recv                                                                   
2023-08-01 08:00:00.002933848+00:00 2023-08-01 08:00:00.002768384+00:00   
2023-08-01 08:00:00.003713709+00:00 2023-08-01 08:00:00.003548245+00:00   
2023-08-01 08:00:00.003780052+00:00 2023-08-01 08:00:00.003615699+00:00   
2023-08-01 08:00:00.004599586+00:00 2023-08-01 08:00:00.004434419+00:00   
2023-08-01 08:00:00.004685512+00:00 2023-08-01 08:00:00.004518771+00:00   

                                     rtype  publisher_id  instrument_id  \
ts_recv                                                                   
2023-08-01 08:00:00.002933848+00:00      1            

In [ ]:
import numpy as np

# 1. Filter for TRADES only (Action 'T')
# In MBP-1, 'T' indicates an execution (a trade)
trades = df[df['action'] == 'T'].copy()

# 2. Filter for Regular Trading Hours (09:30 - 16:00)
trades = trades.between_time('09:30', '16:00')

# 3. Calculate the Midpoint from the prevailing Best Bid/Offer
trades['midpoint'] = (trades['bid_px_00'] + trades['ask_px_00']) / 2

# 4. Apply the 'Quote Rule' (Lee-Ready Step 1)
trades['side_lr'] = 0  # Initialize a new column
trades.loc[trades['price'] > trades['midpoint'], 'side_lr'] = 1  # Buy
trades.loc[trades['price'] < trades['midpoint'], 'side_lr'] = -1 # Sell

# 5. Apply the 'Tick Rule' (Lee-Ready Step 2 for midpoint trades)
# If price is exactly at midpoint, look at the price change from the previous trade
trades['price_diff'] = trades['price'].diff()
midpoint_trades = (trades['side_lr'] == 0)

trades.loc[midpoint_trades & (trades['price_diff'] > 0), 'side_lr'] = 1
trades.loc[midpoint_trades & (trades['price_diff'] < 0), 'side_lr'] = -1

# 6. Recursive Rule (If price didn't move, use the previous classification)
trades['side_lr'] = trades['side_lr'].replace(0, np.nan).ffill()

# 7. Calculate Signed Volume (This is a key input for Phase 2)
trades['signed_vol'] = trades['side_lr'] * trades['size']

print(f"Processed {len(trades)} trades.")
display(trades[['symbol', 'price', 'midpoint', 'side_lr', 'signed_vol']].head(10))

Processed 201027 trades.


,symbol,price,midpoint,side_lr,signed_vol
ts_recv,,,,,
2023-08-01 09:30:02.340963620+00:00,TSLA,265.15,265.210,-1.0,-1.0
2023-08-01 09:30:13.081569401+00:00,MSFT,334.80,334.720,1.0,9.0
2023-08-01 09:30:19.497616245+00:00,TSLA,265.18,265.225,-1.0,-70.0
2023-08-01 09:30:20.589865257+00:00,TSLA,265.18,265.225,-1.0,-200.0
2023-08-01 09:30:20.649599715+00:00,TSLA,265.18,265.225,-1.0,-30.0
2023-08-01 09:30:25.271455302+00:00,AAPL,195.87,195.865,1.0,50.0
2023-08-01 09:30:42.891324879+00:00,TSLA,265.18,265.150,1.0,5.0
2023-08-01 09:30:49.271782180+00:00,TSLA,265.12,265.135,-1.0,-5.0
2023-08-01 09:30:49.271782180+00:00,TSLA,265.12,265.135,-1.0,-15.0
